# Linear Algebra for AI - Week 1
## Vectors, Dot Products, and Cosine Similarity

This notebook covers the fundamental concepts of vector operations used in AI and machine learning:
1. Vector operations and properties
2. Dot products
3. Vector norms (magnitudes)
4. Cosine similarity
5. Practical applications in vector similarity search

Let's start by importing the required libraries:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple

# Set random seed for reproducibility
np.random.seed(42)

## 1. Vector Operations

A vector is an ordered list of numbers. In AI, vectors are fundamental building blocks - they represent features, embeddings, weights, and more.

Let's create some example vectors and explore their properties:

In [ ]:
# Create example vectors
u = np.array([1, 2])
v = np.array([2, 1])

print("Vector u:", u)
print("Vector v:", v)

# Basic vector operations
print("\nVector addition (u + v):", u + v)
print("Vector subtraction (u - v):", u - v)
print("Scalar multiplication (2 * u):", 2 * u)

## 2. Dot Product

The dot product (or inner product) between two vectors is a fundamental operation that measures how similar two vectors are in terms of their direction. 

For vectors a and b, the dot product is defined as:
$a \cdot b = \sum_{i=1}^n a_i b_i$

Let's implement and explore dot products:

In [ ]:
def dot_product(a: np.ndarray, b: np.ndarray) -> float:
    """Compute the dot product between two vectors."""
    return float(np.sum(a * b))

# Calculate dot product of u and v
dot_uv = dot_product(u, v)
print(f"Dot product (u · v): {dot_uv}")

# Verify with numpy's built-in dot product
np_dot = np.dot(u, v)
print(f"NumPy dot product: {np_dot}")

# Test with perpendicular vectors
w = np.array([1, 0])
x = np.array([0, 1])
print(f"\nDot product of perpendicular vectors: {dot_product(w, x)}")  # Should be 0

## 3. Vector Norms

The norm (magnitude) of a vector measures its length. The most common norm is the Euclidean norm (L2 norm):

$\|v\| = \sqrt{\sum_{i=1}^n v_i^2}$

Let's implement vector normalization:

In [ ]:
def normalize(v: np.ndarray) -> np.ndarray:
    """Normalize a vector to unit length."""
    norm = np.linalg.norm(v)
    return v / (norm + 1e-12)  # Add small epsilon to prevent division by zero

# Calculate norms
norm_u = np.linalg.norm(u)
norm_v = np.linalg.norm(v)

print(f"Norm of u: {norm_u:.4f}")
print(f"Norm of v: {norm_v:.4f}")

# Normalize vectors
u_norm = normalize(u)
v_norm = normalize(v)

print(f"\nNormalized u: {u_norm}")
print(f"Normalized v: {v_norm}")

# Verify that normalized vectors have unit length
print(f"\nLength of normalized u: {np.linalg.norm(u_norm):.4f}")
print(f"Length of normalized v: {np.linalg.norm(v_norm):.4f}")

## 4. Cosine Similarity

Cosine similarity measures the cosine of the angle between two vectors. It's defined as:

$\cos(\theta) = \frac{a \cdot b}{\|a\| \|b\|}$

This metric is widely used in AI for:
- Finding similar documents/embeddings
- Measuring semantic similarity
- Nearest neighbor search

Let's implement cosine similarity and test it:

In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Compute cosine similarity between two vectors."""
    return float(dot_product(normalize(a), normalize(b)))

# Test with our vectors u and v
cos_sim_uv = cosine_similarity(u, v)
print(f"Cosine similarity between u and v: {cos_sim_uv:.4f}")

# Test with perpendicular vectors (should be 0)
cos_sim_wx = cosine_similarity(w, x)
print(f"Cosine similarity between perpendicular vectors: {cos_sim_wx:.4f}")

# Test with parallel vectors (should be 1)
cos_sim_uu = cosine_similarity(u, u)
print(f"Cosine similarity between parallel vectors: {cos_sim_uu:.4f}")

# Test with opposite vectors (should be -1)
cos_sim_uNegu = cosine_similarity(u, -u)
print(f"Cosine similarity between opposite vectors: {cos_sim_uNegu:.4f}")

## 5. Practical Application: Vector Similarity Search

Now let's build a simple vector memory system that can:
1. Store vectors with associated IDs
2. Find the most similar vectors using cosine similarity
3. Compute recall@k metrics

This is a fundamental building block for:
- Semantic search
- Recommendation systems
- Memory retrieval in AI systems

In [ ]:
class VectorMemory:
    def __init__(self):
        self.ids: List[str] = []
        self.vectors: List[np.ndarray] = []
        
    def add(self, id_: str, vector: np.ndarray) -> None:
        """Add a vector to memory."""
        self.ids.append(id_)
        self.vectors.append(normalize(vector))
        
    def query(self, query_vector: np.ndarray, top_k: int = 5) -> List[Tuple[str, float]]:
        """Find top-k most similar vectors."""
        query_norm = normalize(query_vector)
        similarities = [cosine_similarity(query_norm, v) for v in self.vectors]
        
        # Get indices of top-k similarities
        top_indices = np.argsort(similarities)[-top_k:][::-1]
        
        # Return (id, similarity) pairs
        return [(self.ids[i], similarities[i]) for i in top_indices]

# Create test data
memory = VectorMemory()
dim = 64  # Dimension of our vectors

# Add 100 random vectors
for i in range(100):
    vector = np.random.randn(dim)  # Random normal distribution
    memory.add(f"vec_{i}", vector)

# Query with a random vector
query = np.random.randn(dim)
results = memory.query(query, top_k=5)

print("Top 5 most similar vectors:")
for id_, sim in results:
    print(f"{id_}: {sim:.4f}")

## 6. Exercises

Now try these exercises to reinforce your understanding:

1. Compute the angle between vectors u and v using the cosine similarity
2. Create a visualization of vector similarity using a heatmap
3. Implement recall@k evaluation for the vector memory system

Let's solve them together:

In [ ]:
# Exercise 1: Compute angle between vectors
cos_angle = cosine_similarity(u, v)
angle_rad = np.arccos(cos_angle)
angle_deg = np.degrees(angle_rad)

print(f"Angle between u and v: {angle_deg:.2f} degrees")

# Exercise 2: Visualize vector similarities
n_vectors = 10
random_vectors = [np.random.randn(dim) for _ in range(n_vectors)]
similarity_matrix = np.zeros((n_vectors, n_vectors))

for i in range(n_vectors):
    for j in range(n_vectors):
        similarity_matrix[i, j] = cosine_similarity(random_vectors[i], random_vectors[j])

plt.figure(figsize=(8, 6))
plt.imshow(similarity_matrix, cmap='RdYlBu')
plt.colorbar(label='Cosine Similarity')
plt.title('Vector Similarity Heatmap')
plt.xlabel('Vector Index')
plt.ylabel('Vector Index')
plt.show()

# Exercise 3: Implement recall@k evaluation
def evaluate_recall_at_k(memory: VectorMemory, queries: List[np.ndarray], 
                        true_neighbors: List[List[str]], k: int) -> float:
    """Compute recall@k for a set of queries."""
    total_recall = 0.0
    
    for query, true_ids in zip(queries, true_neighbors):
        # Get top-k results
        results = memory.query(query, top_k=k)
        retrieved_ids = {id_ for id_, _ in results}
        
        # Compute recall for this query
        n_relevant = len(set(true_ids[:k]) & retrieved_ids)
        recall = n_relevant / min(k, len(true_ids))
        total_recall += recall
    
    return total_recall / len(queries)

# Create test scenario
n_test = 10
test_queries = [np.random.randn(dim) for _ in range(n_test)]
# For this example, we'll use random "true" neighbors
true_neighbors = [[f"vec_{j}" for j in range(5)] for _ in range(n_test)]

recall = evaluate_recall_at_k(memory, test_queries, true_neighbors, k=5)
print(f"\nRecall@5: {recall:.4f}")

## Summary

In this notebook, we covered:
1. Basic vector operations and their properties
2. Dot products and their geometric interpretation
3. Vector norms and normalization
4. Cosine similarity and its applications
5. A practical implementation of vector similarity search
6. Evaluation metrics for similarity search (recall@k)

Key takeaways:
- Cosine similarity is bounded between -1 and 1
- Normalized vectors have length 1
- Vector similarity search is fundamental to many AI applications
- Recall@k measures retrieval performance

Next steps:
1. Experiment with different similarity metrics
2. Implement efficient similarity search using approximate methods
3. Apply these concepts to real-world embeddings from language models
4. Explore dimensionality reduction techniques like PCA

In [ ]:
"""
ASTRA Linear Algebra Notebook - Week 1
Vectors, Dot Products, Cosine Similarity
"""